In [22]:
import os
import torch
import mlflow
import evaluate
import numpy as np 
from datasets import load_dataset
from transformers import (
    TrainingArguments, 
    Trainer, 
    DistilBertForSequenceClassification, 
    DistilBertTokenizer, 
    DataCollatorWithPadding, 
    EvalPrediction
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

In [23]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://fsn1.your-objectstorage.com"

# Train Model

In [24]:
# mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.autolog()   

if not mlflow.get_experiment_by_name(name="Distilbert Fine Tuning"):
    print("Creating model...")
    mlflow.create_experiment(name="Distilbert Fine Tuning")


distillbert_experiment = mlflow.get_experiment_by_name(name="Distilbert Fine Tuning")

2026/05/15 04:09:21 INFO mlflow.tracking.fluent: Autologging successfully enabled for transformers.


Creating model...


In [25]:
if torch.cuda.is_available():
    print("Running training with on CUDA!")

Running training with on CUDA!


In [26]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [27]:
def tokenize(batch):
    tokens =  tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
    )

    tokens['labels'] = batch['label']

    return tokens

dataset = load_dataset("imdb", cache_dir="data/cache", split="train")
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.train_test_split(test_size=0.2)

In [28]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
args = TrainingArguments(
    output_dir="models/distilbert",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    dataloader_pin_memory=False,
    seed=42,
    logging_dir="logs/"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [33]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred: EvalPrediction):
    predictions, labels = eval_pred
    
    # Convert logits to predicted class indices
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate individual metrics
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted')
    precision = precision_metric.compute(predictions=predictions, references=labels, average='weighted')
    recall = recall_metric.compute(predictions=predictions, references=labels, average='weighted')
    
    return {
        'accuracy': accuracy['accuracy'],
        'f1': f1['f1'],
        'precision': precision['precision'],
        'recall': recall['recall']
    }

In [34]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=dataset["train"], 
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [35]:
with mlflow.start_run(experiment_id=distillbert_experiment.experiment_id):
    trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.243462,0.220748,0.913800,0.913788,0.913982,0.913800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🏃 View run whimsical-stag-326 at: http://localhost:5000/#/experiments/3/runs/23262da1a2fa4485b07f837886cfcba6
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [36]:
trainer.save_model()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [37]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.243462,0.220748,1,0.913800,0.913788,0.913982,0.913800


{'eval_loss': 0.2207481861114502,
 'eval_accuracy': 0.9138,
 'eval_f1': 0.9137881196322177,
 'eval_precision': 0.9139816936120742,
 'eval_recall': 0.9138}